In [ ]:
from utilities import initialize_with_module

await initialize_with_module('AITW-darcy_infiltration/1.0.0-foss-2023b', '2023.06')

In [ ]:
import utilities
from common import DARCY_DEFAULTS
from aitw_darcy.cli import infiltration

widget_darcy, darcy_event = utilities.click_cli_widget(
    infiltration,
    override_defaults=DARCY_DEFAULTS,
    title='Darcy Infiltration',
    button_description='Run Darcy Infiltration',
    button_running_text='Running...',
)

display(widget_darcy)

In [ ]:
# Rewrite of the above cell with autodetection of which folders contains the images
import re
import os
import json
import ipywidgets as wdg

rgx_output = re.compile(r"(\S+).json")

title = wdg.HTML(value="<h2>Darcy Infiltration Results</h2>")

def find_outputs(base_path="../results/darcy_results/"):
    outputs = []
    if not os.path.exists(base_path):
        return outputs
    for path in os.listdir(base_path):
        if not os.path.isfile(os.path.join(base_path, path)):
            continue
        match = rgx_output.match(path)
        if not match:
            continue
        label, = match.groups()
        outputs.append((label, os.path.join(base_path, path)))

    return outputs

# Start with a choice widget to select which output to visualize
output_selector = wdg.Dropdown(
    options=[],
    description='Select Output:',
    disabled=False,
    style={'description_width': '150px'},
)
output_selector.layout.width = '500px'

def update_output_options(*args):
    """Update the options of the output selector based on the available outputs in the results folder."""
    outputs = find_outputs()
    options = []
    for label, path in sorted(outputs):
        with open(os.path.join(path), 'r') as f:
            params = json.load(f)
        input_params = params['input_parameters']
        dx = input_params['xmax'] - input_params['xmin']
        dy = input_params['ymax'] - input_params['ymin']
        dz = input_params['zmax'] - input_params['zmin']
        nx = input_params['nx']
        ny = input_params['ny']
        nz = input_params['nz']
        porosity = input_params['porosity']
        options.append((
            f"{label}: ({nx}x{ny}x{nz}) in ({dx:.3f}x{dy:.3f}x{dz:.3f}), porosity={porosity:.3f}",
            path
        ))
    output_selector.options = options
    if output_selector.value is None:
        output_selector.value = options[0][1] if options else None

# Observe changes in the common event to update the output options whenever a new microstructure is generated
try:
    darcy_event.observe(update_output_options, names='value')
except Exception as e:
    print("common-event not defined.")
    print("Re-launch this cell after executing the previous one to enable automatic updates of the dropdown choices.")


output_area = wdg.Output()


def display_sat_time(*args):
    """Display the selected image based on the selected output, toggle state, and slider value."""
    selected_output = output_selector.value
    if not selected_output:
        return

    with open(os.path.join(selected_output), 'r') as f:
        params = json.load(f)
    sat_time = params.get('sat_time', None)
    input_params = params.get('input_parameters', None)

    with output_area:
        output_area.clear_output(wait=True)
        if sat_time is not None:
            print(f"Selected Output: {selected_output}")
            print(f"Input Parameters:")
            print(json.dumps(input_params, indent=4))
            print('*' * 40)
            print(f"Sat Time: {sat_time:.4f}")
            print('*' * 40)

# Observe changes in the toggle and slider to update the displayed image
output_selector.observe(display_sat_time, names='value')

# Initial update of the output options and image options
update_output_options()

display(title, output_selector, output_area)